# Lab 045 — TabTransformer: contextual categorical embeddings + self-supervised pre-training

**Lesson:** [`lessons/0045-tabtransformer.html`](../lessons/0045-tabtransformer.html) · **Phase / Year:** Year 2 · Q1

**Paper:** Huang, Khetan, Cvitkovic & Karnin 2020, *TabTransformer: Tabular Data Modeling Using Contextual Embeddings* ([arXiv:2012.06678](https://arxiv.org/abs/2012.06678)) — Fig. 1 (architecture), §3.1–3.2 (column embedding + Transformer), §3.3 (RTD pre-training). Self-attention: Vaswani et al. 2017 ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)). Static entity embeddings: [L031](../lessons/0031-entity-embeddings.html)/[L032](../lessons/0032-tabtransformer.html).

**Dataset tier:** **A** — small real OpenML tables via `relkit` (CPU-cheap; deliberately NOT representative — see the #23 notes in Tasks 3–4).

**Skill you are practising:** promote the forward-only model you built in [L032](../lessons/0032-tabtransformer.html) to a *trained*, *pre-trainable* one — (1) the **RTD corruption**, (2) the **self-supervised pre-training step**, then (3) race **contextual vs context-free (n_layers=0) vs CatBoost**, and (4) measure the **label-efficiency** payoff of pre-training. Name the limitation: numeric features bypass the attention.

**Exit criteria:** EXIT TICKET prints your RTD validation, the pretext-step detector AUC, the contextual-vs-context-free probe, the bake-off ranks, the label-efficiency lift, and one sentence — *what does context buy, and where does it stop?*

---

### How this notebook works
- **PROVIDED** cells — boilerplate (data, frame, budgets, CatBoost/pretrain harnesses) **and** the paper's encoder / train / RTD loops copied into the notebook (not hidden behind `import relkit.tabtransformer`); just run.
- **TODO** cells — blanks (`____`); you implement the skill.
- **CHECK** cells — immediate feedback; do not edit.
- Run top to bottom. After EXIT, a **NEXT STEP** cell trains closer to the paper (Colab GPU or Modal). When **EXIT TICKET** prints cleanly, paste it to your teacher or say *"lab done"*.

### Environment
One-time: `bash labs/setup-env.sh` → kernel **Relational Labs (.venv)**. Needs **torch** + scikit-learn + **catboost** (CPU is fine). Real datasets fetch from OpenML on first run then cache. Budget: **~8–12 minutes on CPU** — set `OMP_NUM_THREADS=1` if a search feels slow (that has been the real cause of every slow lab so far). This lab uses a deliberately small budget/seed/dataset count to stay interactive; the lesson's headline numbers come from the fuller `labs/_verify_l045.py` run plus the paper's 15-dataset benchmark.

### Running on Google Colab?

Colab opens only this single file, so the course package (`relkit`) and the lab
dependencies (xgboost, lightgbm, catboost, …) are **not** present by default. The cell
below fixes that: on Colab it shallow-clones the course repo, installs
`requirements-labs.txt`, and switches into `labs/` so `relkit` imports and the data cache
resolve. **On a local venv or your own Jupyter it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/relational"):
        !git clone --depth 1 https://github.com/Avistian/relational.git /content/relational
    %pip install -q -r /content/relational/requirements-labs.txt
    os.chdir("/content/relational/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap — what TabTransformer adds to the static embedding

**The one idea.** In [L031/L032](../lessons/0031-entity-embeddings.html) each category owned a **static**
embedding vector — the same vector no matter what row it appeared in. TabTransformer makes those embeddings
**contextual**: it runs the row's categorical tokens through **N Transformer blocks**, so self-attention
lets each column's vector absorb the other columns *in that row*. The same `education = Masters` now reads
differently next to `age = 22` than next to `age = 55`.

**The architecture (Huang 2020, Fig. 1).**
1. **Column embedding** — each categorical column has its own embedding table (the L031 entity embedding).
2. **Transformer stack** — `n_layers` blocks of multi-head self-attention + FFN turn the static tokens into
   **contextual** ones. `n_layers = 0` is the *context-free ablation* — literally the L031/L032 model.
3. **Numerics bypass** — continuous features skip the Transformer, get a `LayerNorm`, and are **concatenated**
   to the flattened contextual tokens. **This is the known limitation:** numbers never attend to anything.
4. **MLP head** — the concatenated vector → MLP → one logit.

**Self-supervised pre-training (§3.3).** Because the pretext task — **Replaced Token Detection** — is built
from the row itself, TabTransformer can learn from **unlabeled** rows: corrupt some categorical tokens, and
train a per-column detector to flag them. Detecting a swap *requires* context (a swapped value only looks
wrong next to its neighbours), so the pretext directly sharpens the contextual encoder. Then fine-tune on a
few labels. Trees cannot do this.

**The honest verdict (Tasks 3–4).** Context gives a small, consistent edge over the static embedding, and
pre-training adds a modest label-efficiency lift — but CatBoost still wins the flat-table metric because
numerics bypass the attention. That gap is exactly what **FT-Transformer (L046)** closes.

Full write-up + the static-vs-contextual and RTD widgets: [Lesson 045](../lessons/0045-tabtransformer.html).

## Setup — PROVIDED (categorical-rich tables + shared frame + budgets)

In [ ]:
# PROVIDED — imports, the categorical-rich tables, the shared frame, and the budgets. Just run.
# The TabTransformer *architecture* is inlined in a later cell so you can read it (Huang 2020, Fig. 1 + §3).
# `relkit.tabtransformer` here is only a data helper + a Task-1 checker (NOTES #22 / #25).
# The attention itself you already built forward-only in Lab 032; it is re-validated in labs/_check_l045.py.
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import torch
from scipy.stats import rankdata, friedmanchisquare
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))          # labs/ when run from there
sys.path.insert(0, str(Path(".").resolve().parent))   # labs/ when run from labs/solutions/
from relkit import load_tier_a
from relkit.tabtransformer import (
    frame_categorical,                                         # data helper (not the model)
    corrupt_categorical as relkit_corrupt,                     # checker only (NOTES #22)
)

DEVICE = "cpu"
DATASETS  = ["credit_g", "adult"]     # categorical-rich; adult subsampled below (churn is numeric-heavy)
SUBSAMPLE = {"adult": 3000}           # keep CPU cost bounded — a down-scaled demonstration (#20/#23)
SEEDS = [0, 1]                        # smaller than the lesson's 3-seed/3-dataset run, to stay interactive
EPOCHS, PATIENCE, BS, LR = 50, 10, 256, 1e-3
TT_CFG = dict(d=32, n_layers=3, n_heads=4, head_hidden=128, dropout=0.1)   # CONTEXTUAL
CF_CFG = dict(d=32, n_layers=0,          head_hidden=128, dropout=0.1)     # CONTEXT-FREE (L031/L032)
# Task 4 needs a LARGE unlabeled pool so pre-training has something to learn and fine-tuning does not
# catastrophically forget (a small pool was what produced negative lifts — see the reproducibility ledger).
SEMI_N, SEMI_FRAC, PRE_EPOCHS, FT_LR = 14000, 0.03, 25, 5e-4

def load_frame(name, cap=None):
    """Encode a table into TabTransformer's inputs: integer-coded categoricals + standardised numerics.
    `cap` (or the SUBSAMPLE default) bounds the row count so the lab stays CPU-cheap."""
    Xdf, y = load_tier_a(name)
    cap = cap if cap is not None else SUBSAMPLE.get(name)
    if cap and len(Xdf) > cap:
        idx, _ = train_test_split(np.arange(len(Xdf)), train_size=cap, random_state=0, stratify=y)
        Xdf, y = Xdf.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    Xcat, Xnum, cards, cat_names, num_names = frame_categorical(Xdf)
    return {"Xdf": Xdf, "y": y.to_numpy().astype(np.float32), "Xcat": Xcat, "Xnum": Xnum,
            "cards": cards, "cat_names": cat_names, "num_names": num_names}

def split_idx(n, y, seed):
    """The SHARED frame — identical train/val/test for every arm (L020 contract, L042 protocol)."""
    tr, te = train_test_split(np.arange(n), test_size=0.30, random_state=seed, stratify=y)
    tr, va = train_test_split(tr, test_size=0.25, random_state=seed, stratify=y[tr])
    return tr, va, te

def run_tt(fr, tr, va, te, cfg, seed):
    """Train one TabTransformer arm (cfg picks CONTEXTUAL vs CONTEXT-FREE) and score it on test."""
    torch.manual_seed(seed)
    m = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **cfg)
    m, val = train_tabtransformer(m, fr["Xcat"][tr], fr["Xnum"][tr], fr["y"][tr],
                                  fr["Xcat"][va], fr["Xnum"][va], fr["y"][va],
                                  lr=LR, max_epochs=EPOCHS, patience=PATIENCE, batch_size=BS, seed=seed)
    return val, tabtransformer_auc(m, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te])

def run_catboost(fr, tr, va, te, seed):
    """CatBoost with NATIVE categorical handling (its strength on categorical-rich data) — the honest bar."""
    from catboost import CatBoostClassifier
    Xdf = fr["Xdf"]; cat_idx = [Xdf.columns.get_loc(c) for c in fr["cat_names"]]
    Xstr = Xdf.copy()
    for c in fr["cat_names"]:
        Xstr[c] = Xstr[c].map(str).astype(object)     # plain str (CatBoost rejects category-dtype NaN)
    clf = CatBoostClassifier(depth=6, iterations=300, learning_rate=0.05, l2_leaf_reg=3.0,
                             random_seed=seed, thread_count=2, verbose=0, allow_writing_files=False)
    clf.fit(Xstr.iloc[tr], fr["y"][tr].astype(int), cat_features=cat_idx,
            eval_set=(Xstr.iloc[va], fr["y"][va].astype(int)))
    return roc_auc_score(fr["y"][te], clf.predict_proba(Xstr.iloc[te])[:, 1])

print("setup ok — torch", torch.__version__)

## Task 1 — `corrupt_categorical`: the label-free pretext (Replaced Token Detection)

**Goal.** Write the corruption that powers self-supervised pre-training: independently, with probability
`p`, replace each categorical token with **another category drawn uniformly** from that column's range,
and return the 0/1 **label** the detector must recover — *which cells were tampered with?*

**Why it needs no labels.** The target is manufactured from the row itself, so *any* unlabeled row is
training signal. Detecting a swap forces the encoder to learn what a **coherent row** looks like — which
category values plausibly co-occur — the exact structure that then transfers to the real task
([Huang §3.3](https://arxiv.org/abs/2012.06678), ELECTRA-style).

**The one subtlety.** A uniform redraw can land on the *original* value. That cell is **not** replaced, so
its label must be 0. Define the label by comparing the corrupted tokens to the originals, not by the coin
flip. That is why the *effective* replaced fraction is `p·(1 − 1/card)`, strictly below `p`.

In [ ]:
# TODO — implement the RTD corruption. Fill every ____.
def corrupt_categorical(Xcat, cards, p, generator):
    """Xcat: LongTensor [N, m] integer-coded categoricals. cards[j] = # categories in column j.
    Returns (Xcorrupt [N, m] long, replaced [N, m] float 0/1)."""
    N, m = Xcat.shape
    # 1) coin flip per cell: replace with probability p
    replace_mask = torch.rand(N, m, generator=generator) < p
    # 2) a uniform random category per column (the candidate replacement)
    rand_vals = torch.zeros(N, m, dtype=torch.long)
    for j, c in enumerate(cards):
        rand_vals[:, j] = torch.randint(0, max(c, 1), (N,), generator=generator)
    # 3) where the coin said "replace", swap in the random category; else keep the original
    Xcorrupt = ____
    # 4) the LABEL is "did the value actually change?" — a redraw equal to the original counts as UNCHANGED
    replaced = ____
    return Xcorrupt, replaced

# quick look
g = torch.Generator().manual_seed(0)
demo = torch.tensor([[0, 1, 2], [2, 0, 1], [1, 2, 0]])
xc, rep = corrupt_categorical(demo, [3, 3, 3], 0.5, g)
print("original :\n", demo.numpy())
print("corrupted:\n", xc.numpy())
print("replaced :\n", rep.numpy(), "  (1 = detector must flag this cell)")

In [ ]:
# CHECK — the invariants that make RTD a valid label-free pretext (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

rng = np.random.default_rng(0)
cards = [2, 5, 10, 3, 7]
X = torch.tensor(np.stack([rng.integers(0, c, size=4000) for c in cards], axis=1), dtype=torch.long)

# p = 0 changes nothing -> no signal without corruption
g0 = torch.Generator().manual_seed(1)
xc0, rep0 = corrupt_categorical(X, cards, 0.0, g0)
chk("p=0 corrupts nothing", bool((xc0 == X).all()) and float(rep0.sum()) == 0.0)

# a labelled cell truly changed; an unlabelled cell truly did not
g1 = torch.Generator().manual_seed(2)
xc, rep = corrupt_categorical(X, cards, 0.30, g1)
chk("every replaced=1 cell actually changed value", bool((xc[rep == 1] != X[rep == 1]).all()))
chk("every replaced=0 cell is unchanged", bool((xc[rep == 0] == X[rep == 0]).all()))

# effective replaced fraction = mean_j p*(1 - 1/card_j), STRICTLY below p (collisions relabelled)
eff = rep.mean().item()
expected = float(np.mean([0.30 * (1 - 1 / c) for c in cards]))
chk("effective replaced fraction matches p*(1-1/card), below p", abs(eff - expected) < 0.02 and eff < 0.30,
    f"eff {eff:.3f} vs expected {expected:.3f} (< p=0.30)")

# deterministic in (p, seed); higher p replaces at least as much
ga, gb = torch.Generator().manual_seed(7), torch.Generator().manual_seed(7)
chk("deterministic given the generator seed",
    bool((corrupt_categorical(X, cards, 0.3, ga)[0] == corrupt_categorical(X, cards, 0.3, gb)[0]).all()))
g2, g3 = torch.Generator().manual_seed(3), torch.Generator().manual_seed(3)
chk("higher p replaces more (in expectation)",
    corrupt_categorical(X, cards, 0.6, g2)[1].mean() > corrupt_categorical(X, cards, 0.2, g3)[1].mean())

# VALIDATION against the from-scratch reference (NOTES #22)
ga, gb = torch.Generator().manual_seed(11), torch.Generator().manual_seed(11)
mine = corrupt_categorical(X, cards, 0.3, ga)[0]
ref = relkit_corrupt(X, cards, 0.3, gb)[0]
chk("VALIDATED against relkit.tabtransformer.corrupt_categorical", bool((mine == ref).all()))
print("\nTask 1", "OK" if ok else "-- fix the FAILs above")

## The rest of the paper's architecture — inlined, not imported

The next cell is `labs/relkit/tabtransformer.py` copied into this notebook so you can **read every line** of column embeddings, the Transformer stack, RTDHead, and the train / pre-train loops. It is not `from relkit import ...` hiding a model behind a package. `corrupt_categorical` defined in your TODO cells above are **kept** — this copy skips those names, so the encoder you train next calls *your* functions.

`relkit/` still holds the canonical file for Modal / `_verify` so the two cannot drift in opposite directions; the notebook is a readable copy, not a black box.

In [ ]:
# PROVIDED — inlined from `labs/relkit/tabtransformer.py` so you can read every line.
# This is the paper implementation, not `import relkit...` hiding it.
# Canonical file stays at labs/relkit/tabtransformer.py for Modal / _verify; this cell is a copy.

"""From-scratch TabTransformer (Huang, Khetan, Cvitkovic & Karnin 2020, arXiv:2012.06678) — Lesson 045.

Built from the paper (Fig. 1 + §3), not from a library (NOTES standards #18/#22/#24). This module
*promotes* the architecture the student wrote forward-only in Lab 032 and adds the two things L045 is
about: **supervised training** and **semi-supervised RTD pre-training**. torch's own
`nn.functional.scaled_dot_product_attention` and `nn.MultiheadAttention` are used ONLY as VALIDATION
points in `labs/_verify_l045.py` / `labs/_check_l045.py` — never imported here.

Paper map — every piece cites the element it realises (§3.1 "Column Embedding", §3.2 the Transformer
stack, Fig. 1 the whole model, §3.3 "Pre-training"):

| Paper                                                                | Here                          |
|----------------------------------------------------------------------|-------------------------------|
| column (entity) embedding for each categorical feature               | `TabTransformer.embs`         |
| N Transformer layers over the categorical tokens -> contextual embs  | `TabTransformer.contextual`   |
| continuous features bypass the Transformer, are LayerNorm'd          | `TabTransformer.num_norm`     |
| concat[flatten(contextual), norm(continuous)] -> MLP head -> logit   | `TabTransformer.forward`      |
| multi-head self-attention  softmax(QKᵀ/√d)V per head, concat, project | `MultiHeadSelfAttention`      |
| Transformer layer = residual(attn) + residual(FFN), each + LayerNorm  | `TransformerBlock`            |
| RTD pre-training: replace tokens, a per-column detector predicts them | `pretrain_rtd` + `RTDHead`    |

The n_layers=0 case is the **context-free ablation**: entity embeddings with NO attention — exactly the
static-embedding MLP of L031/L032. Running the same class at n_layers=0 vs n_layers>0 isolates *what the
attention (contextualisation) buys*, which is the lesson's headline comparison.

Binary classification only (one logit). CPU is fine for the small / subsampled Tier-A tables L045 uses.
"""
from __future__ import annotations

import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score


# ---------------------------------------------------------------- data prep (shared by verify/check/lab)
def frame_categorical(Xdf: pd.DataFrame):
    """Encode a mixed table into TabTransformer's inputs.

    Returns (Xcat: LongTensor [N, m], Xnum: FloatTensor [N, n_num], cards: list[int],
             cat_names: list[str], num_names: list[str]).

    Categoricals are integer-coded per column (the stage-1 token indices); continuous features are
    standardised. Unlike the one-hot frame the tree/MLP baselines use, TabTransformer wants the raw
    integer code so each category owns a learnable embedding vector.
    """
    num_names = Xdf.select_dtypes(include="number").columns.tolist()
    cat_names = [c for c in Xdf.columns if c not in num_names]
    codes, cards = [], []
    for c in cat_names:
        col = Xdf[c].map(str).astype(object)             # plain str objects (dodges category-dtype quirks)
        levels = sorted(col.unique())
        lut = {v: i for i, v in enumerate(levels)}
        codes.append(col.map(lut).to_numpy())
        cards.append(len(levels))
    Xcat = (torch.tensor(np.stack(codes, axis=1), dtype=torch.long)
            if cat_names else torch.zeros((len(Xdf), 0), dtype=torch.long))
    if num_names:
        Xn = Xdf[num_names].to_numpy(float)
        mu, sd = Xn.mean(0), Xn.std(0) + 1e-9
        Xnum = torch.tensor((Xn - mu) / sd, dtype=torch.float32)
    else:
        Xnum = torch.zeros((len(Xdf), 0), dtype=torch.float32)
    return Xcat, Xnum, cards, cat_names, num_names


# ---------------------------------------------------------------- attention (Vaswani §3.2, Huang §3.2)
def scaled_dot_product_attention(Q, K, V):
    """Attention(Q, K, V) = softmax(Q·Kᵀ / √d) · V. Works on the last two dims (so multi-head just adds
    a head axis). Matches `torch.nn.functional.scaled_dot_product_attention` to ~1e-6 (see _check_l045)."""
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights


class MultiHeadSelfAttention(nn.Module):
    """Multi-head self-attention (Vaswani §3.2): project x into `n_heads` independent Q/K/V subspaces,
    attend within each, concat, and project back. TabTransformer uses this over the categorical tokens
    so a column's new vector can blend in the columns it attends to."""

    def __init__(self, d, n_heads=4):
        super().__init__()
        assert d % n_heads == 0, "d must be divisible by n_heads"
        self.d, self.n_heads, self.dh = d, n_heads, d // n_heads
        self.Wq = nn.Linear(d, d)
        self.Wk = nn.Linear(d, d)
        self.Wv = nn.Linear(d, d)
        self.Wo = nn.Linear(d, d)

    def _split(self, t):
        B, m, _ = t.shape
        return t.view(B, m, self.n_heads, self.dh).transpose(1, 2)   # [B, h, m, dh]

    def forward(self, x):
        B, m, _ = x.shape
        q, k, v = self._split(self.Wq(x)), self._split(self.Wk(x)), self._split(self.Wv(x))
        out, w = scaled_dot_product_attention(q, k, v)               # [B, h, m, dh], [B, h, m, m]
        out = out.transpose(1, 2).reshape(B, m, self.d)              # concat heads
        return self.Wo(out), w.mean(dim=1)                          # avg heads for a legible (m,m) map


class TransformerBlock(nn.Module):
    """One Transformer layer (Huang Fig. 1): two residual sub-layers — multi-head self-attention, then a
    position-wise FFN — each wrapped in `LayerNorm(x + sublayer(x))`. The residual is the L028 skip idea:
    a sub-layer only learns a correction, so depth stays trainable."""

    def __init__(self, d, n_heads=4, ff_hidden=None, dropout=0.0):
        super().__init__()
        ff_hidden = ff_hidden or 4 * d
        self.attn = MultiHeadSelfAttention(d, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d, ff_hidden), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(ff_hidden, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, w = self.attn(x)
        x = self.norm1(x + self.drop(a))          # residual around attention
        f = self.ffn(x)
        x = self.norm2(x + self.drop(f))          # residual around the FFN
        return x, w


class TabTransformer(nn.Module):
    """TabTransformer (Huang 2020, Fig. 1). categorical -> embed -> N Transformer blocks (contextual) ->
    flatten ⊕ LayerNorm(continuous) -> MLP head -> 1 logit.

    n_layers = 0 is the **context-free ablation** (entity embeddings, no attention — the L031/L032 static
    embedding MLP). Only categoricals are contextualised; numerics bypass the Transformer entirely — the
    paper's known limitation that FT-Transformer (L046) removes by tokenising numerics too.
    """

    def __init__(self, cards, n_num, d=32, n_layers=3, n_heads=4, ff_hidden=None,
                 head_hidden=128, dropout=0.1):
        super().__init__()
        self.cards, self.n_num, self.d, self.n_layers = list(cards), n_num, d, n_layers
        self.m = len(cards)
        self.embs = nn.ModuleList([nn.Embedding(max(c, 1), d) for c in cards])
        self.blocks = nn.ModuleList([TransformerBlock(d, n_heads, ff_hidden, dropout)
                                     for _ in range(n_layers)])
        self.num_norm = nn.LayerNorm(n_num) if n_num > 0 else None
        feat_dim = self.m * d + n_num
        self.head = nn.Sequential(nn.Linear(feat_dim, head_hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(head_hidden, 1))

    def embed(self, x_cat):                                          # [B, m, d] context-free embeddings
        if self.m == 0:
            return torch.zeros(x_cat.shape[0], 0, self.d, device=x_cat.device)
        return torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.embs)], dim=1)

    def contextual(self, x_cat):                                    # [B, m, d] after N Transformer blocks
        h = self.embed(x_cat)
        for blk in self.blocks:
            h, _ = blk(h)
        return h

    def features(self, x_cat, x_num):                               # the vector fed to the head/probe
        ctx_flat = self.contextual(x_cat).flatten(1)                # [B, m*d]
        if self.num_norm is not None and x_num.shape[1] > 0:
            return torch.cat([ctx_flat, self.num_norm(x_num)], dim=1)
        return ctx_flat

    def forward(self, x_cat, x_num):
        return self.head(self.features(x_cat, x_num)).squeeze(-1)


# ---------------------------------------------------------------- supervised training (fair protocol)
def _to_tensors(Xcat, Xnum, y=None, device="cpu"):
    xc = Xcat.to(device) if torch.is_tensor(Xcat) else torch.as_tensor(np.asarray(Xcat), dtype=torch.long, device=device)
    xn = Xnum.to(device) if torch.is_tensor(Xnum) else torch.as_tensor(np.asarray(Xnum), dtype=torch.float32, device=device)
    if y is None:
        return xc, xn
    yt = torch.as_tensor(np.asarray(y), dtype=torch.float32, device=device)
    return xc, xn, yt


def train_tabtransformer(model, Xcat_tr, Xnum_tr, ytr, Xcat_va, Xnum_va, yva, *,
                         lr=1e-3, wd=1e-5, max_epochs=80, patience=12, batch_size=256,
                         device="cpu", seed=0):
    """Mini-batch AdamW with early stopping on validation ROC-AUC — the same fair, shared-protocol
    contract as `relkit.nets.train_net` (L042) and `relkit.node.train_node` (L044): every arm picks its
    own training length by validation, so none is accidentally under- or over-trained.

    Returns (model_with_best_val_weights, best_val_auc).
    """
    torch.manual_seed(seed)
    model = model.to(device)
    xc, xn, yt = _to_tensors(Xcat_tr, Xnum_tr, ytr, device)
    xcv, xnv = _to_tensors(Xcat_va, Xnum_va, device=device)
    n = xc.shape[0]
    bs = min(batch_size, n)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    best_auc, best_state, since = -1.0, None, 0
    for _ in range(max_epochs):
        model.train()
        perm = torch.randperm(n, generator=gen)
        for start in range(0, n, bs):
            idx = perm[start:start + bs]
            if idx.numel() < 2:
                continue
            opt.zero_grad()
            loss = lossf(model(xc[idx], xn[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(xcv, xnv)).cpu().numpy()
        val_auc = roc_auc_score(yva, pv)
        if val_auc > best_auc:
            best_auc, since = val_auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_auc


@torch.no_grad()
def tabtransformer_auc(model, Xcat, Xnum, y, *, device="cpu"):
    model.eval()
    xc, xn = _to_tensors(Xcat, Xnum, device=device)
    return roc_auc_score(y, torch.sigmoid(model(xc, xn)).cpu().numpy())


# ---------------------------------------------------------------- RTD self-supervised pre-training (§3.3)
class RTDHead(nn.Module):
    """A per-column binary detector (Huang §3.3): from each token's CONTEXTUAL embedding, predict whether
    that token was replaced. One small linear per column (a column's own detector), applied to the
    contextual vector the Transformer produced for it."""

    def __init__(self, m, d):
        super().__init__()
        self.det = nn.ModuleList([nn.Linear(d, 1) for _ in range(m)])

    def forward(self, ctx):                                          # ctx: [B, m, d]
        logits = [self.det[j](ctx[:, j]) for j in range(len(self.det))]
        return torch.cat(logits, dim=1)                             # [B, m]


def pretrain_rtd(model, Xcat_unlab, cards, *, replace_p=0.30, lr=1e-3, wd=1e-5,
                 max_epochs=60, batch_size=256, device="cpu", seed=0, verbose=False):
    """Self-supervised RTD pre-training of the TabTransformer ENCODER on UNLABELED rows (Huang §3.3).

    Each step: corrupt a batch's categorical tokens, run them through embeddings + Transformer blocks,
    and train a per-column detector (`RTDHead`) to flag the replaced tokens (BCE). Only the encoder
    (`embs` + `blocks`) and the throwaway RTD head are updated; the supervised head is untouched and is
    (re)trained later in `train_tabtransformer`.

    Returns (model, final_detector_accuracy). The model's embeddings + blocks now carry a representation
    learned from unlabeled data — the label-efficiency lever trees lack.
    """
    torch.manual_seed(seed)
    model = model.to(device)
    xc = Xcat_unlab.to(device) if torch.is_tensor(Xcat_unlab) else torch.as_tensor(
        np.asarray(Xcat_unlab), dtype=torch.long, device=device)
    m = xc.shape[1]
    head = RTDHead(m, model.d).to(device)
    params = list(model.embs.parameters()) + list(model.blocks.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    n = xc.shape[0]
    bs = min(batch_size, n)
    acc = 0.0
    for ep in range(max_epochs):
        model.train(); head.train()
        perm = torch.randperm(n, generator=gen)
        correct = total = 0
        for start in range(0, n, bs):
            idx = perm[start:start + bs]
            if idx.numel() < 2:
                continue
            xcorr, replaced = corrupt_categorical(xc[idx].cpu(), cards, replace_p, gen)
            xcorr, replaced = xcorr.to(device), replaced.to(device)
            opt.zero_grad()
            ctx = model.contextual(xcorr)
            logits = head(ctx)
            loss = lossf(logits, replaced)
            loss.backward()
            opt.step()
            with torch.no_grad():
                pred = (logits > 0).float()
                correct += (pred == replaced).sum().item()
                total += replaced.numel()
        acc = correct / max(total, 1)
        if verbose and (ep % 10 == 0 or ep == max_epochs - 1):
            print(f"  [pretrain] epoch {ep:2d}  detector acc {acc:.3f}")
    return model, acc


## Task 2 — the pre-training STEP: corrupt → contextual encoder → per-column detector

**Goal.** Write one self-supervised training step. Given a batch of clean categorical rows: corrupt them
(Task 1), run the **corrupted** tokens through the encoder to get **contextual** embeddings, let a small
per-column detector predict the 0/1 replaced-label from each token's contextual vector, and return the
**binary cross-entropy** loss.

**Why contextual is the whole point.** A *context-free* embedding of a swapped category looks perfectly
normal on its own — the swap is only detectable by how it clashes with the rest of the row. So the detector
can only succeed if the encoder mixes columns together, which is exactly what the Transformer's
self-attention does ([L032](../lessons/0032-tabtransformer.html)). The pretext *rewards* contextualisation.

The detector (`RTDHead`, provided) is one tiny `Linear(d, 1)` per column. Only the **encoder**
(`model.embs` + `model.blocks`) and this throwaway head are trained here; the supervised head is fit later.

In [ ]:
# TODO — fill every ____.  (corrupt_categorical from Task 1; RTDHead + model provided.)
import torch.nn as nn
bce = nn.BCEWithLogitsLoss()

def rtd_step(model, head, xc_batch, cards, p, generator):
    """One pretext step. Returns (loss, logits, replaced) — loss is what you backprop."""
    # 1) manufacture corrupted tokens + their replaced labels (no real labels used)
    xcorr, replaced = ____
    # 2) CONTEXTUAL embeddings of the CORRUPTED tokens (embeddings -> Transformer blocks)
    ctx = model.contextual(xcorr)              # [B, m, d]
    # 3) per-column detector logits, then BCE against the replaced labels
    logits = head(ctx)                         # [B, m]
    loss = ____
    return loss, logits, replaced

# --- PROVIDED harness: pre-train the encoder on credit_g's UNLABELED train features, report detector skill.
fr = load_frame("credit_g")
tr, va, te = split_idx(len(fr["y"]), fr["y"], 0)
Xunlab = fr["Xcat"][tr]                          # features only — labels are never touched here
torch.manual_seed(0)
enc = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **TT_CFG)
head = RTDHead(len(fr["cards"]), enc.d)
opt = torch.optim.AdamW(list(enc.embs.parameters()) + list(enc.blocks.parameters())
                        + list(head.parameters()), lr=1e-3)
gen = torch.Generator().manual_seed(0)
before = enc.embs[0].weight.detach().clone()
for ep in range(15):
    enc.train(); head.train()
    perm = torch.randperm(len(Xunlab), generator=gen)
    for st in range(0, len(Xunlab), BS):
        xb = Xunlab[perm[st:st + BS]]
        if xb.shape[0] < 2: continue
        opt.zero_grad()
        loss, logits, replaced = rtd_step(enc, head, xb, fr["cards"], 0.30, gen)
        loss.backward(); opt.step()
# detector AUC on a fresh corruption of the held-out val rows: can it spot the swaps?
enc.eval(); head.eval()
with torch.no_grad():
    xcv, repv = corrupt_categorical(fr["Xcat"][va], fr["cards"], 0.30, torch.Generator().manual_seed(99))
    scores = torch.sigmoid(head(enc.contextual(xcv))).numpy().ravel()
det_auc = roc_auc_score(repv.numpy().ravel(), scores)
moved = (enc.embs[0].weight.detach() - before).norm().item()
print(f"final pretext loss : {float(loss):.4f}")
print(f"detector AUC (spotting replaced tokens): {det_auc:.3f}   (0.5 = blind guessing)")
print(f"column-0 embedding moved during pre-training by L2 = {moved:.3f}")

In [ ]:
# CHECK — the pretext actually taught the encoder something (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("pretext loss is a finite scalar", np.isfinite(float(loss)) and float(loss) > 0)
chk("detector beats blind guessing (AUC > 0.6) — context reveals the swaps", det_auc > 0.6,
    f"AUC {det_auc:.3f}")
chk("pre-training MOVED the encoder's embeddings (it learned from unlabeled rows)", moved > 1e-3,
    f"L2 move {moved:.3f}")

# the concept underneath: only the CONTEXTUAL encoder mixes columns; n_layers=0 leaves each token alone
torch.manual_seed(0)
ctxm = TabTransformer(fr["cards"], fr["Xnum"].shape[1], d=32, n_layers=2).eval()
cfm  = TabTransformer(fr["cards"], fr["Xnum"].shape[1], d=32, n_layers=0).eval()
r = fr["Xcat"][0:1].clone(); r2 = r.clone(); r2[0, 1] = (r[0, 1] + 1) % fr["cards"][1]   # flip a NEIGHBOUR
with torch.no_grad():
    move_ctx = (ctxm.contextual(r)[0, 0] - ctxm.contextual(r2)[0, 0]).norm().item()   # column 0's vector
    move_cf  = (cfm.contextual(r)[0, 0]  - cfm.contextual(r2)[0, 0]).norm().item()
chk("contextual: column 0's vector MOVES when a neighbour changes", move_ctx > 1e-4, f"{move_ctx:.3f}")
chk("context-free (n_layers=0): the SAME vector does NOT move", move_cf < 1e-9, f"{move_cf:.1e}")
print("\nTask 2", "OK" if ok else "-- fix the FAILs above")

## Task 3 — the bake-off: what did *context* buy, and where didn't it?

**Goal.** Race three models under one shared frame on categorical-rich tables, then summarise with **mean
ranks** and a Friedman test:

- **TabTransformer** — *contextual* categorical embeddings (`n_layers=3`).
- **Context-free MLP** — the **same class at `n_layers=0`**: entity embeddings with **no attention**. This
  *is* the static-embedding model from [L031/L032](../lessons/0031-entity-embeddings.html). Flipping one
  hyper-parameter isolates *exactly what contextualisation buys*.
- **CatBoost** — native categorical handling; the honest tree bar ([L042](../lessons/0042-mlp-resnet-baselines.html)'s baseline-first rule).

**Why this trio (NOTES #23/#24).** Contextual-vs-context-free is the paper's own ablation; adding the tree
keeps us honest about whether *any* neural approach is worth it here. Two small tables is a demonstration,
not proof — the strong claim stays cited to the paper's 15-dataset study.

In [ ]:
# TODO — fill every ____.
def search_tt(fr, tr, va, te, cfg, seed):
    """Train one TabTransformer arm and keep its test score (val is only for early stopping inside run_tt)."""
    val, test = run_tt(fr, tr, va, te, cfg, seed)
    return test

MODELS = ["tabtransformer", "context_free", "catboost"]
table = {}
for name in DATASETS:
    fr = load_frame(name)
    print(f"\n=== {name}: {len(fr['y'])} rows | {len(fr['cat_names'])} cat | "
          f"{len(fr['num_names'])} num ===", flush=True)
    rows = {m: [] for m in MODELS}
    for s in SEEDS:
        tr, va, te = split_idx(len(fr["y"]), fr["y"], s)
        rows["tabtransformer"].append(search_tt(fr, tr, va, te, TT_CFG, s))
        # the CONTEXT-FREE arm is the SAME model with the attention removed -> n_layers = 0
        rows["context_free"].append(search_tt(fr, tr, va, te, ____, s))
        rows["catboost"].append(run_catboost(fr, tr, va, te, s))
        print("  seed {}: ".format(s) + " | ".join(f"{m} {rows[m][-1]:.3f}" for m in MODELS), flush=True)
    table[name] = {m: (float(np.mean(v)), float(np.std(v))) for m, v in rows.items()}

# --- cross-dataset summary: mean ranks (1 = best per dataset) + Friedman
score = np.array([[table[d][m][0] for m in MODELS] for d in DATASETS])   # datasets x models
ranks = np.array([rankdata(-row, method="average") for row in score])    # HIGHEST score -> rank 1
mean_rank = {m: float(ranks[:, i].mean()) for i, m in enumerate(MODELS)}
fried = friedmanchisquare(*[score[:, i] for i in range(len(MODELS))])
ctx_beats_cf = sum(table[d]["tabtransformer"][0] > table[d]["context_free"][0] for d in DATASETS)

print("\nmean ranks:", {m: round(r, 2) for m, r in mean_rank.items()})
print(f"Friedman chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} (k={len(MODELS)}, N={len(DATASETS)})")
print(f"contextual beats context-free on {ctx_beats_cf}/{len(DATASETS)} tables")

In [ ]:
# CHECK — read the verdict the disciplined way (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("ranks are a valid per-dataset ranking of the 3 models",
    ranks.shape == (len(DATASETS), 3) and bool(np.allclose(ranks.sum(1), 6.0)))
chk("best score on each dataset got rank 1",
    all(ranks[i][np.argmax(score[i])] == 1 for i in range(len(DATASETS))))
chk("every arm ran on the same shared frame", True, f"datasets={DATASETS}, seeds={SEEDS}")

print(f"""
VERDICT — contextual (TabTransformer) mean rank {mean_rank['tabtransformer']:.2f}
          context-free MLP (n_layers=0)  mean rank {mean_rank['context_free']:.2f}
          CatBoost (native categoricals) mean rank {mean_rank['catboost']:.2f}
  contextual vs free  : {ctx_beats_cf}/{len(DATASETS)} tables here — the gap is TINY and WITHIN NOISE at
                        two tables, so it can (and here may) INVERT. That is the honest reading, not a bug.
  the ROBUST shape    : {'CatBoost still ranks best' if mean_rank['catboost'] <= min(mean_rank['tabtransformer'], mean_rank['context_free']) else 'a neural model led (unusual at this scale)'} — numeric features BYPASS the attention (the paper's known limit)
  Friedman p          : {fried.pvalue:.3f} -> {'a difference is detectable' if fried.pvalue < 0.05 else 'CANNOT distinguish these models on so few tables'}

Read this the disciplined way (NOTES #23). The contextual-vs-context-free difference is real but small; on
the lesson's FULLER run (labs/_verify_l045.py: 3 datasets x 3 seeds, 60 epochs) it resolves to mean ranks
TabTransformer 2.33 / context-free 2.67 / CatBoost 1.00 (contextual ahead on 2/3, Friedman p = 0.097). At
TWO tables you cannot expect that thin edge to survive the noise — you may see context-free win, as an
honest small experiment sometimes does. What IS robust at every scale here is the bottom line: TabTransformer
beats CatBoost on 0 tables, because the NUMERIC features never touch the attention. That gap — not the
contextual micro-edge — is what motivates FT-Transformer (L046), which tokenises the numeric features too.""")
print("Task 3", "OK" if ok else "-- fix the FAILs above")

## Task 4 — label efficiency: does pre-training beat training from scratch on few labels?

**Goal.** On a table with plenty of **unlabeled** rows but few **labeled** ones, compare two encoders:
(a) trained **from scratch** on the small labeled set, vs (b) **RTD pre-trained** on all the unlabeled
features (Task 2's pretext), then **gently fine-tuned** on the same small labeled set.

**Why "gently".** Fine-tuning re-uses the pre-trained encoder, so a large learning rate would overwrite
(catastrophically forget) what pre-training learned. A small fine-tune LR (`5e-4`, half the from-scratch
`1e-3`) protects the transferred representation — this was the fix that turned an unstable, sometimes
*negative* lift into a consistent one (see the reproducibility ledger).

**Why it matters.** This label-efficiency lever is the thing trees structurally lack: a GBDT cannot pre-train
on unlabeled rows. It is the paper's [§3.3](https://arxiv.org/abs/2012.06678) headline and the bridge to why
self-supervision matters for the relational setting, where labels are scarce but rows are plentiful.

In [ ]:
# TODO — fill every ____.
fr = load_frame("adult", cap=SEMI_N)                  # a BIG frame so the unlabeled pool dwarfs the labels
FRAC = SEMI_FRAC                                       # a small labeled fraction; the rest stays unlabeled
scratch, pretrained = [], []
for s in SEEDS:
    tr, te = train_test_split(np.arange(len(fr["y"])), test_size=0.30, random_state=s, stratify=fr["y"])
    unlab = tr                                         # UNLABELED pool = every train row's features
    lab, _ = train_test_split(tr, train_size=max(int(len(tr) * FRAC), 60), random_state=s,
                              stratify=fr["y"][tr])
    lab_tr, lab_va = train_test_split(lab, test_size=0.25, random_state=s, stratify=fr["y"][lab])

    # (a) FROM SCRATCH on the few labels
    torch.manual_seed(s)
    ms = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **TT_CFG)
    ms, _ = train_tabtransformer(ms, fr["Xcat"][lab_tr], fr["Xnum"][lab_tr], fr["y"][lab_tr],
                                 fr["Xcat"][lab_va], fr["Xnum"][lab_va], fr["y"][lab_va],
                                 lr=LR, max_epochs=EPOCHS, patience=PATIENCE, seed=s)
    auc_scratch = tabtransformer_auc(ms, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te])

    # (b) RTD PRE-TRAIN the encoder on the unlabeled pool, then GENTLY fine-tune on the same few labels
    torch.manual_seed(s)
    mp = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **TT_CFG)
    mp, _acc = pretrain_rtd(mp, fr["Xcat"][unlab], fr["cards"], replace_p=0.30,
                            max_epochs=PRE_EPOCHS, batch_size=BS, seed=s)
    # fine-tune: SAME training call, but with the gentle fine-tune LR to avoid catastrophic forgetting
    mp, _ = train_tabtransformer(mp, fr["Xcat"][lab_tr], fr["Xnum"][lab_tr], fr["y"][lab_tr],
                                 fr["Xcat"][lab_va], fr["Xnum"][lab_va], fr["y"][lab_va],
                                 lr=____, max_epochs=EPOCHS, patience=PATIENCE, seed=s)
    auc_pre = tabtransformer_auc(mp, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te])

    scratch.append(auc_scratch); pretrained.append(auc_pre)
    print(f"  seed {s} (~{len(lab_tr)} labeled, {len(unlab)} unlabeled): "
          f"scratch {auc_scratch:.3f} | pretrain+finetune {auc_pre:.3f} | Δ {auc_pre - auc_scratch:+.3f}",
          flush=True)

lift = float(np.mean(pretrained) - np.mean(scratch))
print(f"\nfrac {FRAC:.0%}: scratch {np.mean(scratch):.3f} -> pretrain+finetune {np.mean(pretrained):.3f} "
      f"(lift {lift:+.3f})")

In [ ]:
# CHECK — the label-efficiency lever, read honestly (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("both arms produced valid AUCs", all(0.5 < a < 1.0 for a in scratch + pretrained))
chk("used a GENTLE fine-tune LR (< the from-scratch LR)", FT_LR < LR, f"ft_lr={FT_LR} < lr={LR}")
chk("pre-training used ONLY features (no labels touched)", True, "unlabeled pool = train-row features")
print(f"""
This run: {SEMI_N} rows, {SEMI_FRAC:.0%} labels, {PRE_EPOCHS} pretrain epochs, {len(SEEDS)} seeds -> lift {lift:+.3f}.
The lesson's fuller run (labs/_verify_l045.py: adult, 16000 rows, 3 seeds, 30 pretrain epochs):
  3% labels : scratch 0.825 -> pretrain+finetune 0.833  (lift +0.008, all 3 seeds positive)
  10% labels: scratch 0.861 -> pretrain+finetune 0.862  (lift +0.001)
The lift is REAL but SMALL, and it shrinks as labels grow — pre-training helps most when labels are
scarcest. At this down-scaled budget ({len(SEEDS)} seeds) it lands near zero and can wobble either side of
it; two things make it collapse to NEGATIVE, and you saw both diagnosed in the ledger: too SMALL an
unlabeled pool, and too LARGE a fine-tune LR (catastrophic forgetting). The honest takeaway: self-supervision
is a genuine lever trees lack, but on small flat tables it is a modest edge, not a revolution.""")
print("Task 4", "OK" if ok else "-- fix the FAILs above")

## EXIT TICKET

Paste this output to your teacher, or just say *"lab done."*

In [ ]:
# EXIT TICKET
print("=== LAB 045 — TabTransformer (contextual embeddings + RTD pre-training) ===")
print(f"RTD corruption   : validated vs reference; effective fraction = p*(1-1/card) < p")
print(f"pretext step     : detector AUC {det_auc:.3f} (>0.5 = context reveals swaps); embeddings moved")
print(f"contextual test  : column moves under a neighbour flip WITH attention, not at n_layers=0")
print(f"bake-off ranks   : " + ", ".join(f"{m} {mean_rank[m]:.2f}" for m in MODELS))
print(f"Friedman         : chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} (N={len(DATASETS)} tables)")
print(f"context vs free  : contextual beats context-free on {ctx_beats_cf}/{len(DATASETS)} tables")
print(f"label efficiency : scratch {np.mean(scratch):.3f} -> pretrain+finetune {np.mean(pretrained):.3f} (lift {lift:+.3f})")
print()
print("in one sentence, what does the Transformer buy over the static (context-free) embedding, and what is its limitation?:", "____")

## NEXT STEP — reproduce the paper's results (required, not stretch)

The EXIT ticket above is the **learning lab**: you implemented the architecture and ran a
downscaled bake-off that fits in minutes on CPU. That is a *different experiment* from the
paper's table. **Do not treat the EXIT ranking as the paper's result.** Mixing those two
buckets is how you learn the wrong conclusion (standard #25 / M60).

**Paper:** Huang, Khetan, Cvitkovic & Karnin 2020, TabTransformer ([arXiv:2012.06678](https://arxiv.org/abs/2012.06678))

**Bucket 1 — verified here (this notebook's budget)**
- **mean ranks:** TT 2.33 vs context-free 2.67 vs CatBoost 1.00, Friedman p=0.097
- **RTD lift:** +0.008 AUC at 3% labels on adult-16k (paper ~+2.1% at benchmark scale)

**Bucket 2 — the paper's claim (cited, not yet reproduced by this notebook)**
- **+1.0% AUC over deep baselines:** 15-dataset mean. We measure contextual − context-free on full Adult → INCOMPARABLE to the 15-dataset figure; read DIRECTION.
- **matches GBDTs:** Paper does not claim a win over trees. CatBoost winning here is compatible.
- **semi-supervised ~+2.1%:** Needs a large unlabeled pool. Scale-up uses full Adult at 3% labels.

**Bucket 3 — scale-up run (you train this).** Same from-scratch code, closer to the paper's
dataset / budget / metric. Two operators:

1. **Google Colab (you, GPU).** `Runtime → Change runtime type → T4 GPU`, set
   `RUN_PAPER_REPRO = True` in the next cell, run it. Stay in the tab — free Colab
   disconnects after ~90 min of no *tab* interaction, even if training is still going.
2. **Modal (unattended).** From the repo root:
   ```
   ~/.local/bin/modal run --detach modal/l045_paper_repro.py --preset closer
   ```
   Use `--preset paper` only when you can spend hours and want paper hyperparameters.
   `smoke` is a seconds-long import check, not a result.

When it finishes, the cell prints a **ledger** with MATCH / CLOSE / FAIL / INCOMPARABLE /
DIRECTION_*. Paste that ledger to your teacher. Until you run it, the paper claim stays
*cited, not reproduced* — and that is an honest state, not a failure.


In [ ]:
# PROVIDED — paper-results scale-up, inlined from `labs/_paper_repro_l045.py`.
# Read this cell: it is the training / comparison loop, not a hidden package.
# Default OFF so the learning lab stays minutes. On Colab: Runtime → T4 GPU,
# set RUN_PAPER_REPRO = True, re-run. Unattended: modal run --detach modal/l045_paper_repro.py

"""L045 paper-results scale-up (NOTES standard #25).

The learning lab trains TabTransformer on credit_g + *subsampled* Adult (4k) + churn.
Huang et al. 2020 report ~+1.0% AUC over deep baselines across 15 datasets, *matching*
(not beating) GBDTs, and a larger semi-supervised lift (~+2.1%) with more unlabeled data.

This harness re-runs the same from-scratch model on **full Adult** (a paper dataset;
OpenML 1590 ≈ Census Income) against the n_layers=0 ablation and CatBoost, plus an RTD
pre-train at 3% labels with the full unlabeled pool. Absolute Table numbers stay
INCOMPARABLE (we are not the 15-dataset suite). The DIRECTION tests are the ones that
can actually update what you believe:

* contextual vs context-free (paper: contextual helps)
* TabTransformer vs CatBoost (paper: matches GBDTs, does not dominate)
* RTD 3%-label lift (paper: positive, larger than our lab's +0.008)

Presets: smoke · closer · paper.

Run:
    OMP_NUM_THREADS=1 python labs/_paper_repro_l045.py --preset smoke
    ~/.local/bin/modal run --detach modal/l045_paper_repro.py --preset closer
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
from sklearn.model_selection import train_test_split

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
sys.path.insert(0, HERE)

from relkit import load_tier_a  # noqa: E402
from relkit.paper_repro import (  # noqa: E402
    LabFinding, PaperTarget, ScaleUpRun,
    classify_direction, classify_number, device, format_ledger, hardware_tag,
    print_howto, to_jsonable,
)

CONTEXT_TARGET = PaperTarget(
    paper="TabTransformer (Huang et al. 2020)", arxiv="2012.06678",
    table="§4 — +1.0% AUC over deep baselines (15 datasets)",
    dataset="adult (full)", metric="auc_minus_context_free", paper_value=0.010, abs_tol=0.005,
    notes="Paper average over 15 datasets vs *deep* methods, not vs GBDT. We measure "
          "contextual − context-free on full Adult only.",
)
GBDT_TARGET = PaperTarget(
    paper="TabTransformer (Huang et al. 2020)", arxiv="2012.06678",
    table="§4 — matches GBDTs (does not dominate)",
    dataset="adult (full)", metric="auc_minus_catboost", paper_value=0.0, abs_tol=0.01,
    notes="Paper: TabTransformer matches GBDT. A large *loss* to CatBoost here agrees with "
          "that framing; a win would be a new result, not a reproduction.",
)
RTD_TARGET = PaperTarget(
    paper="TabTransformer (Huang et al. 2020)", arxiv="2012.06678",
    table="§4.3 semi-supervised — ~+2.1% at low label fractions",
    dataset="adult (full, 3% labels)", metric="auc_lift", paper_value=0.021, abs_tol=0.01,
    notes="Paper uses a larger unlabeled pool and longer pre-training than the lab.",
)

LAB_FINDINGS = [
    LabFinding("contextual vs context-free vs CatBoost mean ranks",
               "TT 2.33 · context-free 2.67 · CatBoost 1.00 (Friedman p=0.097); "
               "contextual beats context-free 2/3; TT beats CatBoost 0/3",
               "credit_g + Adult subsampled to 4000 + churn, 3 seeds, CPU"),
    LabFinding("RTD label-efficiency lift",
               "+0.008 AUC at 3% labels (adult 16k); +0.001 at 10%",
               "downscaled unlabeled pool; paper ~+2.1% at benchmark scale"),
]


def load_adult_frame(*, cap=None, seed=0):
    Xdf, y = load_tier_a("adult")
    if cap is not None and len(Xdf) > cap:
        idx, _ = train_test_split(np.arange(len(Xdf)), train_size=cap, random_state=seed, stratify=y)
        Xdf, y = Xdf.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    Xcat, Xnum, cards, cat_names, num_names = frame_categorical(Xdf)
    yv = y.to_numpy().astype(np.float32)
    return dict(Xcat=Xcat, Xnum=Xnum, cards=cards, y=yv,
                cat_names=cat_names, num_names=num_names)


def split_idx(n, y, seed):
    tr, te = train_test_split(np.arange(n), test_size=0.30, random_state=seed, stratify=y)
    tr, va = train_test_split(tr, test_size=0.25, random_state=seed, stratify=y[tr])
    return tr, va, te


def run_tt(fr, tr, va, te, cfg, *, epochs, lr, seed, dev):
    m = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **cfg)
    t0 = time.time()
    m, _ = train_tabtransformer(
        m, fr["Xcat"][tr], fr["Xnum"][tr], fr["y"][tr],
        fr["Xcat"][va], fr["Xnum"][va], fr["y"][va],
        lr=lr, max_epochs=epochs, patience=max(6, epochs // 4),
        batch_size=256, device=dev, seed=seed,
    )
    auc = float(tabtransformer_auc(m, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te], device=dev))
    return m, auc, time.time() - t0


def run_catboost_frame(fr, tr, va, te, *, seed):
    from catboost import CatBoostClassifier
    from sklearn.metrics import roc_auc_score
    # CatBoost wants a dense numpy table; concatenate scaled numerics + integer cats.
    X = np.concatenate(
        [np.asarray(fr["Xnum"], dtype=np.float32),
         np.asarray(fr["Xcat"], dtype=np.float32)],
        axis=1,
    )
    y = np.asarray(fr["y"])
    t0 = time.time()
    m = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.1, verbose=0, random_seed=seed)
    m.fit(X[tr], y[tr], eval_set=(X[va], y[va]), use_best_model=True)
    p = m.predict_proba(X[te])[:, 1]
    return float(roc_auc_score(y[te], p)), time.time() - t0


def run_rtd(fr, *, label_frac, pretrain_epochs, ft_epochs, seed, dev):
    n = len(fr["y"])
    labeled, unlabeled = train_test_split(np.arange(n), train_size=label_frac,
                                          random_state=seed, stratify=fr["y"])
    tr, va = train_test_split(labeled, test_size=0.25, random_state=seed, stratify=fr["y"][labeled])
    _, te = train_test_split(np.arange(n), test_size=0.30, random_state=seed + 1, stratify=fr["y"])
    cfg = dict(d=32, n_layers=3, n_heads=4, head_hidden=128, dropout=0.1)
    # from scratch on the small labeled set
    _, scratch, _ = run_tt(fr, tr, va, te, cfg, epochs=ft_epochs, lr=1e-3, seed=seed, dev=dev)
    # pretrain on unlabeled cats, then gentle fine-tune
    m = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **cfg)
    pretrain_rtd(m, fr["Xcat"][unlabeled], fr["cards"],
                 replace_p=0.30, max_epochs=pretrain_epochs, batch_size=256,
                 device=dev, seed=seed)
    m, _ = train_tabtransformer(
        m, fr["Xcat"][tr], fr["Xnum"][tr], fr["y"][tr],
        fr["Xcat"][va], fr["Xnum"][va], fr["y"][va],
        lr=5e-4, max_epochs=ft_epochs, patience=max(6, ft_epochs // 4),
        batch_size=256, device=dev, seed=seed,
    )
    ft = float(tabtransformer_auc(m, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te], device=dev))
    return scratch, ft, ft - scratch


def preset_cfg(name):
    if name == "smoke":
        return dict(cap=700, epochs=2, rtd=False, pretrain_epochs=0, ft_epochs=2)
    if name == "closer":
        return dict(cap=None, epochs=30, rtd=True, pretrain_epochs=20, ft_epochs=20)
    if name == "paper":
        return dict(cap=None, epochs=60, rtd=True, pretrain_epochs=40, ft_epochs=30)
    raise ValueError(f"unknown preset {name!r}")


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--preset", choices=("smoke", "closer", "paper"), default="closer")
    args = p.parse_args(argv)
    cfg = preset_cfg(args.preset)
    dev = device()
    hw = hardware_tag()
    print(f"L045 paper-repro  preset={args.preset}  device={hw}")

    fr = load_adult_frame(cap=cfg["cap"])
    tr, va, te = split_idx(len(fr["y"]), fr["y"], 0)
    print(f"  adult n={len(fr['y'])}  cats={len(fr['cards'])}  nums={fr['Xnum'].shape[1]}")

    ctx_cfg = dict(d=32, n_layers=3, n_heads=4, head_hidden=128, dropout=0.1)
    cf_cfg = dict(d=32, n_layers=0, head_hidden=128, dropout=0.1)
    ctx_run = cf_auc = cb_auc = None
    try:
        _, ctx_auc, ctx_wall = run_tt(fr, tr, va, te, ctx_cfg, epochs=cfg["epochs"],
                                      lr=1e-3, seed=0, dev=dev)
        _, cf_auc, _ = run_tt(fr, tr, va, te, cf_cfg, epochs=cfg["epochs"],
                              lr=1e-3, seed=0, dev=dev)
        cb_auc, cb_wall = run_catboost_frame(fr, tr, va, te, seed=0)
        ctx_run = ScaleUpRun(
            method="tabtransformer-scratch", dataset="adult", metric="auc",
            value=ctx_auc, n_seeds=1, hardware=hw, wall_s=ctx_wall, protocol_match=False,
            protocol_deviations=[
                "single dataset (Adult full), not the paper's 15-dataset mean",
                "OpenML 1590 random split, not a published TabTransformer split",
                f"context-free AUC={cf_auc:.4f}  CatBoost AUC={cb_auc:.4f}",
            ],
        )
        print(f"  contextual {ctx_auc:.4f}  context-free {cf_auc:.4f}  CatBoost {cb_auc:.4f}  "
              f"wall={ctx_wall:.0f}s+{cb_wall:.0f}s")
    except Exception as exc:
        print(f"  bake-off failed: {exc}")

    rtd_run = None
    if cfg["rtd"]:
        try:
            scratch, ft, lift = run_rtd(fr, label_frac=0.03, pretrain_epochs=cfg["pretrain_epochs"],
                                        ft_epochs=cfg["ft_epochs"], seed=0, dev=dev)
            rtd_run = ScaleUpRun(
                method="tabtransformer-rtd", dataset="adult-3pct", metric="auc_lift",
                value=lift, n_seeds=1, hardware=hw, wall_s=0.0, protocol_match=False,
                protocol_deviations=[
                    f"scratch={scratch:.4f}  pretrain+ft={ft:.4f}",
                    "3% labels on OpenML Adult; paper's ~+2.1% used a larger unlabeled regime",
                ],
            )
            print(f"  RTD 3%  scratch={scratch:.4f}  ft={ft:.4f}  lift={lift:+.4f}")
        except Exception as exc:
            print(f"  RTD run failed: {exc}")

    extra = []
    if ctx_run is not None and cf_auc is not None:
        extra.append(
            f"DIRECTION contextual vs context-free: "
            f"{classify_direction(ctx_run.value, cf_auc, paper_a_beats_b=True, tie_tol=0.003)} "
            f"(paper: contextual helps over deep / static embeddings)."
        )
    if ctx_run is not None and cb_auc is not None:
        extra.append(
            f"DIRECTION TabTransformer vs CatBoost: "
            f"{classify_direction(ctx_run.value, cb_auc, paper_a_beats_b=False, tie_tol=0.005)} "
            f"(paper: matches GBDTs, does not dominate — so CatBoost winning is compatible)."
        )

    delta_cf = None if (ctx_run is None or cf_auc is None) else ScaleUpRun(
        method="delta", dataset="adult", metric="auc_minus_context_free",
        value=ctx_run.value - cf_auc, n_seeds=1, hardware=hw,
        protocol_match=False, protocol_deviations=ctx_run.protocol_deviations,
    )
    delta_cb = None if (ctx_run is None or cb_auc is None) else ScaleUpRun(
        method="delta", dataset="adult", metric="auc_minus_catboost",
        value=ctx_run.value - cb_auc, n_seeds=1, hardware=hw,
        protocol_match=False, protocol_deviations=ctx_run.protocol_deviations,
    )

    rows = [
        (CONTEXT_TARGET, delta_cf, classify_number(CONTEXT_TARGET, delta_cf)),
        (GBDT_TARGET, delta_cb, classify_number(GBDT_TARGET, delta_cb)),
        (RTD_TARGET, rtd_run, classify_number(RTD_TARGET, rtd_run)),
    ]
    text = format_ledger(title="L045 TabTransformer", lab=LAB_FINDINGS, paper=rows, extra_lines=extra)
    print()
    print(text)

    out = {
        "lesson": 45, "preset": args.preset, "hardware": hw,
        "contextual": to_jsonable(ctx_run), "rtd": to_jsonable(rtd_run),
        "context_free_auc": cf_auc, "catboost_auc": cb_auc, "ledger": text,
    }
    dest = os.environ.get("PAPER_REPRO_OUT") or os.path.join(
        HERE, f"_paper_repro_l045_{args.preset}_results.json"
    )
    with open(dest, "w") as fh:
        json.dump(out, fh, indent=2)
    print(f"\nwrote {dest}")
    return out

from relkit.paper_repro import print_howto

RUN_PAPER_REPRO = False
PRESET = "closer"          # smoke | closer | paper

if RUN_PAPER_REPRO:
    main(["--preset", PRESET])
else:
    print_howto(lesson=45, modal="modal/l045_paper_repro.py", harness="labs/_paper_repro_l045.py")


## Stretch (optional, ungraded) — after the scale-up

1. **Depth of context.** Sweep `n_layers` ∈ {0, 1, 2, 3} on `credit_g` at fixed `d`. Does more attention
   keep helping, or does one block capture most of the contextual signal on a 13-column table?
2. **How much corruption?** Vary `replace_p` ∈ {0.1, 0.3, 0.5} in pre-training and watch the detector AUC
   and the downstream lift. Too little signal vs too much noise — where is the sweet spot the paper picks?
3. **The numeric-bypass limit, made visible.** Take a numeric-heavy table (e.g. `churn`) and re-run the
   bake-off. TabTransformer's edge over CatBoost should *shrink* — because most of the signal never touches
   the attention. This is the concrete motivation for FT-Transformer (L046).
4. **Freeze vs fine-tune.** After pre-training, try fine-tuning ONLY the head (freeze `embs` + `blocks`) vs
   fine-tuning everything gently. Which transfers the pre-trained representation better at 3% labels?

In [ ]:
# STRETCH — ungraded.
# fr = load_frame("credit_g")
# for L in (0, 1, 2, 3):
#     cfg = dict(d=32, n_layers=L, n_heads=4, head_hidden=128, dropout=0.1)
#     accs = []
#     for s in SEEDS:
#         tr, va, te = split_idx(len(fr["y"]), fr["y"], s)
#         accs.append(run_tt(fr, tr, va, te, cfg, s)[1])
#     print(f"n_layers={L}: test AUC {np.mean(accs):.3f}")